## speed tests

### TODO:
- compare relative speedup to 2-core size
- compare relative speedup of fns to that of omidi g-tries, see if it outperforms it
- try cython

In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import functions_regular as fnr
import functions_subtrees as fns
import matplotlib.pyplot as plt
import importlib
import numpy.lib.recfunctions as rfn
import cProfile as cp
import random as rnd
import itertools as it
import collections
import math
import time
importlib.reload(fnr)
importlib.reload(fns)

<module 'functions_subtrees' from '/Users/mbpr/Desktop/internships/CU Boulder Summer 2019/Motif discovery/motif-discovery/src/functions_subtrees.py'>

# measure relative speedup on sample of real networks

In [319]:
def add_label(file):
    f = open(file, 'r')
    
    data = f.readlines()
    
    for index, line in enumerate(data):
        
        split = line.split()
        print(index, data[index], split)
        
        if (len(split) == 1):
            temp = data[index].rstrip() + ' ' + data[index]
            data[index] = temp
            
    f = open(file, 'w')
    f.writelines(data)
    

### read and clean up


In [86]:
# convert file format to string  
    
file = "test_data/ColiNet-1.1/coliInterFullVec.txt"
stringGraph = edgeList_to_string(file)
Graph = parseStringToGraph(stringGraph)

stringQuery = '[[0,1],[0,2],[0,3],[3,4],[3,5]]'
Query = parseStringToGraph(stringQuery)


In [87]:
# choose only largest connected component

largest_cc = max(nx.connected_components(Graph), key=len)
x = list(largest_cc)[0]
F = nx.Graph()

for r in largest_cc:

    F.add_node(r)

    for i in largest_cc:
        for j in largest_cc:
            if ((i, j) in Graph.edges()):
                F.add_edge(i, j)
                    
Graph = nx.Graph()
Graph.add_edges_from(F.edges())

In [88]:
# remove self loops
Graph.remove_edges_from(Graph.selfloop_edges())

# reorder labels from 0 to Graph size - 1
F = nx.Graph()
F = fnr.relabelByDegree(Graph)

Graph = nx.Graph()
Graph.add_edges_from(F.edges())
stringGraph = graphToString(Graph)

list(sorted(Graph.nodes()))

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


### plot

In [ ]:
pos = nx.drawing.kamada_kawai_layout(Graph, scale=1)

fig, ax = plt.subplots()
fig.set_size_inches(18, 18)


nodes = nx.draw_networkx_nodes(Graph, pos=pos, node_size = 0, ax = ax)
edges = nx.draw_networkx_edges(Graph, pos=pos, ax = ax)
labels = nx.draw_networkx_labels(Graph, pos=pos)

plt.show()

### compare speed

In [91]:
start_time_fns = time.time()
out_fns = fns.findSubgraphInstances(stringQuery, stringGraph)
delta_fns = time.time() - start_time_fns

start_time_fnr = time.time()
out_fnr = fnr.findSubgraphInstances(Query, Graph)
delta_fnr = time.time() - start_time_fnr  

print(stringQuery)
print('subtrees:', delta_fns), 'regular:', delta_fnr)

[[0,1],[0,2],[0,3],[3,4],[3,5]]
subtrees: 308.6806831359863


In [83]:
outFile = 'test_data/speed_tests/colinet_1_ryser.txt'

write_speed_results(outFile, delta_fnr, delta_fns, stringQuery, 'colinet1.1')

# compare relative speedup to two-core size

In [310]:
G = nx.gnp_random_graph(50, 0.3, seed=0)
G = growOneShell(G, 60, 0)

In [ ]:
pos = nx.drawing.kamada_kawai_layout(G, scale=1)

fig, ax = plt.subplots()
fig.set_size_inches(10, 10)

nodes = nx.draw_networkx_nodes(G, pos=pos, node_size = 50, ax = ax)
edges = nx.draw_networkx_edges(G, pos=pos, ax = ax)

plt.show()

In [ ]:
fns.onionDecompose(G)

In [259]:
attr = nx.get_node_attributes(G, 'coreness')
twoCoreSize = len([i for i in attr.values() if i > 1])
twoCoreSize

25

In [313]:
# generate a sample of graphs with varying two-core sizes
# run both algorithms against them and measure relative speedup
# also generate graphs with fixed 2-core size and varying 1-shell

# query_string: string representing query graph
# fix: speifies whether to fix graph size or core size
# size: size of core if core is fixed, size of graph if graph is fixed
# step: step size of graph/core size sweep
# sample_size: size of sample from which runtimes are averaged
# core_density: probability of an edge between any two vertices in the 2-core
def speed_test_two_core(query_string, fix='graph', size=100, step=10, sample_size=10, core_density=0.5):
    to_return = {}
    
    if(size < 20):
        print('g not large enough. select g larger than 20')
        return
    
    
    for var_size in range(step, int(0.6*size), step):
        x = 100*(var_size-step)/size
        print('progress:', x, '%')
        
        fnr_samples = []
        fns_samples = []

        for seed in range(sample_size):
            if (fix == 'graph'):
                G = nx.gnp_random_graph(var_size, core_density, seed=seed)
                G = growOneShell(G, size, seed+sample_size)

            elif (fix == 'core'):
                G = nx.gnp_random_graph(size, core_density, seed=seed)
                G = growOneShell(G, size+2*var_size, seed+sample_size)
                
            else:
                print('incorrect argument for fix. should be either core or graph.')
                return

            stringG = graphToString(G)

            stringH = query_string
            H = parseStringToGraph(stringH)

            start_time_fns = time.time()
            out_fns = fns.findSubgraphInstances(stringH, stringG)
            delta_fns = time.time() - start_time_fns        

            start_time_fnr = time.time()
            out_fnr = fnr.findSubgraphInstances(H, G, True)
            delta_fnr = time.time() - start_time_fnr

            fnr_samples.append(delta_fnr)
            fns_samples.append(delta_fns)

        to_return[var_size] = {'regular': fnr_samples, 
                               'subtrees': fns_samples}

    return to_return
        

In [314]:
line_graph_string = '[[0,1],[0,2],[0,3],[0,4],[0,5]]'
line_graph_runtimes = speed_test_two_core(line_graph_string, fix='graph', size = 100, sample_size = 1, core_density = 0.3)

progress: 0.0 %
progress: 10.0 %
progress: 20.0 %
progress: 30.0 %
progress: 40.0 %


In [315]:
line_graph_runtimes

{10: {'regular': [3.823058843612671], 'subtrees': [1.5346629619598389]},
 20: {'regular': [18.493573904037476], 'subtrees': [2.702821969985962]},
 30: {'regular': [29.016381978988647], 'subtrees': [9.174164056777954]},
 40: {'regular': [50.24687886238098], 'subtrees': [26.127470016479492]},
 50: {'regular': [106.98596096038818], 'subtrees': [75.3568320274353]}}

# useful functions

In [2]:
def parseStringToGraph(s):
    l = []
    
    if(s[len(s)-1] == ' '):
        s = s[:-1]
    
    if(s[0] == '[' and s[len(s)-1] == ']'):
        s = s[-(len(s)-1):-1]
        start = [key for key, val in enumerate(s) if val == '[']
        end = [key for key, val in enumerate(s) if val == ']']
        comma = [key for key, val in enumerate(s) if (val == ',' and not s[key-1] == ']')]
        
        for i in range(len(start)):
            left = [v for k, v in enumerate(s) if k in range(start[i]+1, comma[i])]
            right = [v for k, v in enumerate(s) if k in range(comma[i]+1, end[i])]
            
            left = int(''.join(left))
            right = int(''.join(right))
            l.append((left, right))
    
    H = nx.Graph()
    H.add_edges_from(l)
    
    return H

def graphToString(G):
    edges = list(G.edges())
    string = '['
    
    for edge in edges:
        to_add = '[' + str(edge[0]) + ',' + str(edge[1]) + '], '
        string += to_add
        
    string = string[:len(string)-2]
    string += ']'
    
    return string

# takes a graph G and "grows" a 1-shell from it until it reaches size size
def growOneShell(G, size, seed):
    K = nx.Graph()
    rnd.seed(seed)
    
    if (size < len(G)):
        print('size too small')
        return
    
    n = len(G)
    m = size - n
    
    weighedNodeList = []
    
    for n in G.nodes():
        mult = [n]*len(G[n])
        weighedNodeList = weighedNodeList + mult
    
    for newNode in range(n, size):
        adjacentNode = rnd.sample(weighedNodeList, 1)
        G.add_edge(newNode, adjacentNode[0])
        
        for n in G.nodes():
            mult = [n]*len(G[n])
            weighedNodeList = weighedNodeList + mult
        
    K.add_edges_from(G.edges())
    return K


def write_speed_results(outFile, delta_fnr, delta_fns, stringQuery, graphName):
    f = open(outFile, 'r')
    old_data = f.readlines()
    
    of = open(outFile, 'w')
    str_subtrees_time = str(delta_fns)
    str_regular_time = str(delta_fnr)
    
    data = old_data + ['query graph: ', stringQuery, ', network: ', graphName + ' ', ', subtrees: ', str_subtrees_time, ', regular: ', str_regular_time]

    of.writelines(data)
    
    
def edgeList_to_string(file):
    f = open(file, 'r')
    data = f.readlines()
    string = '['
    
    for index, line in enumerate(data):
        split = line.split()
        
#         print(split[0], split[1])
        
        to_add = '[' + split[0] + ',' + split[1] + '], '
        string += to_add
        
    string = string[:len(string)-2]
    string += ']'
    
    return string 